In [1]:
import os

#!git clone https://github.com/recsyspolimi/RecSys_Course_AT_PoliMi

!pwd
os.chdir("../RecSys_Course_AT_PoliMi")
!pwd

#!python run_compile_all_cython.py

/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/m2_kfold
/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/RecSys_Course_AT_PoliMi


In [2]:
import os
import time 
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scipy.sparse as sps
import matplotlib.pyplot as pyplot
%matplotlib inline

from sklearn.model_selection import KFold
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from skopt.space import Real, Integer, Categorical
from scipy.sparse import csr_matrix
from Evaluation.Evaluator import EvaluatorHoldout
from HyperparameterTuning.SearchBayesianSkopt import SearchBayesianSkopt

from Recommenders.SLIM.SLIMElasticNetRecommender import SLIMElasticNetRecommender
from Recommenders.EASE_R.EASE_R_Recommender import EASE_R_Recommender
from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender
from Recommenders.MatrixFactorization.IALSRecommender import IALSRecommender
#from Recommenders.KNN.ItemKNNCBFRecommender import ItemKNNCBFRecommender
#from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender
#from Recommenders.KNN.ItemKNNCustomSimilarityRecommender import ItemKNNCustomSimilarityRecommender

from Recommenders.MatrixFactorization.IALSRecommender import FeatureCombinedImplicitALSRecommender


from Recommenders.hybrid.m3_model import TripleIntegratedHierarchicalHybridRecommender
#from Recommenders.hybrid.m2_model import IntegratedHierarchicalHybridRecommender
#from Recommenders.hybrid.SimilarityMergingHybridRecommender import SimilarityMergingHybridRecommender
#from Recommenders.hybrid.LinearHybridRecommender import GeneralizedLinearCoupleHybridRecommender


Tensorflow is not available


In [3]:
df_train = pd.read_csv("data_train.csv")
df_test_user = pd.read_csv("data_target_users_test.csv")

# Training knn costum similarity

In [4]:
SLIM_params = {
    'topK': 625,
    'l1_ratio': 0.09517221375634205,
    'alpha': 0.00228698730766055
}

"""KNN_params = {
    'similarity': 'tversky',
    'topK': 8,
    'shrink': 100,
    'tversky_alpha': 0.18445514996044549,
    'tversky_beta': 1.7490566752549062,
    'feature_weighting': 'TF-IDF',
}"""

IALS_params = {
    'iterations': 135,
    'factors': 87,
    'alpha': 7.762338288061237,
    'regularization': 0.004799745261257595
}

EASE_params = {
    'topK': 1431,
    'l2_norm': 426.57622242296605
}

rp3_params = {
    'alpha': 0.7733352330682174,
    'beta': 0.4139018623121251,
    'topK': 35
}

In [5]:
from scipy.sparse import coo_matrix

#valore 1 per ogni coppia (row, col)
data = [1] * len(df_train)
df_train["row"] = df_train["row"].astype(int)
df_train["col"] = df_train["col"].astype(int)

# matrice COO
URM_all = sp.csr_matrix((data, (df_train["row"], df_train["col"])))


In [6]:
def split_train_in_five_percentage_global_sample(URM_all, train_percentages):
    """
    The function splits an URM in five matrices based on provided percentages.
    :param URM_all: The full URM matrix
    :param train_percentages: A list of percentages (must sum to 1.0)
    :return: A list of 5 sparse matrices
    """

    import numpy as np
    from scipy.sparse import coo_matrix
    from Data_manager.IncrementalSparseMatrix import IncrementalSparseMatrix

    assert len(train_percentages) == 5, "You must provide exactly 5 percentages."
    assert abs(sum(train_percentages) - 1.0) < 1e-6, "Percentages must sum to 1.0."

    num_users, num_items = URM_all.shape

    # Builders for each of the 5 matrices
    builders = [
        IncrementalSparseMatrix(n_rows=num_users, n_cols=num_items, auto_create_col_mapper=False, auto_create_row_mapper=False)
        for _ in range(5)
    ]

    URM_all_coo = coo_matrix(URM_all)

    # Shuffle indices
    indices_for_sampling = np.arange(URM_all.nnz, dtype=np.int32)
    np.random.shuffle(indices_for_sampling)

    # Calculate the number of interactions for each split
    split_sizes = [int(URM_all.nnz * percentage) for percentage in train_percentages]
    cumulative_sizes = np.cumsum(split_sizes)

    # Divide the indices into 5 groups
    indices_splits = [
        indices_for_sampling[cumulative_sizes[i - 1]:cumulative_sizes[i]] if i > 0 else indices_for_sampling[:cumulative_sizes[i]]
        for i in range(5)
    ]

    # Populate the builders
    for i, builder in enumerate(builders):
        builder.add_data_lists(
            URM_all_coo.row[indices_splits[i]],
            URM_all_coo.col[indices_splits[i]],
            URM_all_coo.data[indices_splits[i]],
        )

    # Convert to sparse matrices
    sparse_matrices = [builder.get_SparseMatrix() for builder in builders]

    # Ensure all outputs are in csr_matrix format
    sparse_matrices = [sp.csr_matrix(matrix) for matrix in sparse_matrices]

    return sparse_matrices

In [7]:
train_percentages = [0.2, 0.2, 0.2, 0.2, 0.2]  # Cinque parti uguali

URM_parts = split_train_in_five_percentage_global_sample(URM_all, train_percentages)
URM_parts

[<Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>]

In [8]:
import os
from scipy import sparse

output_folder = "./saved_models/"
urm_folder = "./saved_urm/"

for folder in [output_folder, urm_folder]:
    if not os.path.exists(folder):
        os.makedirs(folder)

prefitted_folds = []

for i in range(5):
    print(f"Fitting fold {i+1}/5...")
    
    urm_path = os.path.join(urm_folder, f"URM_train_fold_{i}.npz")
    
    # 1. Gestione URM Train
    if os.path.exists(urm_path):
        print(f"Loading URM_train for fold {i}...")
        URM_train = sparse.load_npz(urm_path)
    else:
        print(f"Creating and saving URM_train for fold {i}...")
        URM_train = sum(URM_parts[j] for j in range(len(URM_parts)) if j != i)
        sparse.save_npz(urm_path, URM_train)

    URM_test = URM_parts[i]
    evaluator_test = EvaluatorHoldout(URM_test, cutoff_list=[20])

    # 2. Inizializzazione Modelli
    recommender_slim = SLIMElasticNetRecommender(URM_train)
    recommender_ease = EASE_R_Recommender(URM_train)
    recommender_rp3 = RP3betaRecommender(URM_train)
    recommender_ials = FeatureCombinedImplicitALSRecommender(URM_train)

    model_names = {
        "slim": (recommender_slim, SLIM_params),
        "ease": (recommender_ease, EASE_params),
        "rp3": (recommender_rp3, rp3_params),
        "ials": (recommender_ials, IALS_params)
    }

    # 3. Fit o Load dei modelli
    for name, (model, params) in model_names.items():
        file_name = f"{name}_fold_{i}"
        # Verifichiamo se il file del modello esiste (il framework aggiunge solitamente un'estensione o crea una cartella)
        """try:
            model.load_model(output_folder, file_name=file_name)
            print(f"Loaded {name} from disk.")
        except (FileNotFoundError, Exception):"""
        print(f"Fitting {name}...")
        model.fit(**params)
        model.save_model(output_folder, file_name=file_name)
        print(f"Saved {name} to disk.")
        #print(type(recommender_slim.W_sparse), recommender_slim.W_sparse.nnz)

    # 4. Popolamento lista per ottimizzazione/test
    fold_data = {
        "URM_train": URM_train,
        "slim": recommender_slim,
        "ease": recommender_ease,
        "rp3": recommender_rp3,
        "ials": recommender_ials,
        "evaluator": evaluator_test
    }
    
    prefitted_folds.append(fold_data)

print("\nTask completato: tutti i fold sono pronti in memoria.")

print("Pre-training completato.")

Fitting fold 1/5...
Creating and saving URM_train for fold 0...
EvaluatorHoldout: Ignoring 31 ( 0.1%) Users that have less than 1 test interactions
Fitting slim...
SLIMElasticNetRecommender: Processed 4921 (70.6%) in 5.00 min. Items per second: 16.40
SLIMElasticNetRecommender: Processed 6969 (100.0%) in 7.08 min. Items per second: 16.41
SLIMElasticNetRecommender: Saving model in file './saved_models/slim_fold_0'
SLIMElasticNetRecommender: Saving complete
Saved slim to disk.
Fitting ease...
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 12.97 sec
EASE_R_Recommender: Saving model in file './saved_models/ease_fold_0'
EASE_R_Recommender: Saving complete
Saved ease to disk.
Fitting rp3...
RP3betaRecommender: Similarity column 6969 (100.0%), 3433.60 column/sec. Elapsed time 2.03 sec
RP3betaRecommender: Saving model in file './saved_models/rp3_fold_0'
RP3betaRecommender: Saving complete
Saved rp3 to disk.
Fitting ials...
Saved ials to disk.
Fitting fold 2/5

In [ ]:
import time 

class SaveResults(object):
    
    def __init__(self):
        self.results_df = pd.DataFrame(columns=["result", "train_time (min)"])
    
    def __call__(self, optuna_study, optuna_trial):
        hyperparam_dict = optuna_trial.params.copy()
        hyperparam_dict["result"] = optuna_trial.values[0]
        
        # Retrieve the optimal number of epochs and training time from the "user attributes" of the trial
        #hyperparam_dict["epochs"] = optuna_trial.user_attrs["epochs"]
        hyperparam_dict["train_time (min)"] = optuna_trial.user_attrs["train_time (min)"]
        
        self.results_df.loc[len(self.results_df)] = hyperparam_dict
        
        
def objective_function_funksvd(optuna_trial):

    start_time = time.time()
    scores = []
    for fold_data in prefitted_folds:
        
        # Recuperiamo i modelli e i dati dal dizionario
        URM_train = fold_data["URM_train"]
        recommender_slim = fold_data["slim"]
        recommender_ease = fold_data["ease"]
        recommender_rp3 = fold_data["rp3"]
        recommender_ials = fold_data["ials"]
        evaluator_test = fold_data["evaluator"]
        
        
        recommender = TripleIntegratedHierarchicalHybridRecommender(
            URM_train, 
            recommender_slim, 
            recommender_ease,
            recommender_rp3,
            recommender_ials
        )

        alpha=optuna_trial.suggest_float("alpha", 0.1, 0.25)
        beta=optuna_trial.suggest_float("beta", 0.05, 0.2)
        gamma=optuna_trial.suggest_float("gamma", 0.05, 0.2)

        recommender.fit(alpha, beta, gamma)

        result, _ = evaluator_test.evaluateRecommender(recommender)
        #print("prova = ", result["MAP"].values[0])
        #print(i)
        #print(result["RECALL"])
        scores.append(result["RECALL"].values[0])
        #if result["MAP"].values[0] < 0.051:
        #    break
        
    # Add the number of epochs selected by earlystopping as a "user attribute" of the optuna trial
    #epochs = recommender_instance.get_early_stopping_final_epochs_dict()["epochs"]
    #optuna_trial.set_user_attr("epochs", epochs) 
    optuna_trial.set_user_attr("train_time (min)", (time.time() - start_time)/60) 
    print(scores)
    return sum(scores) / len(scores)


In [14]:
import optuna

optuna_study = optuna.create_study(direction="maximize")
        
save_results = SaveResults()
        
optuna_study.optimize(objective_function_funksvd,
                      callbacks=[save_results],
                      n_trials = 100)

[I 2025-12-28 11:32:30,036] A new study created in memory with name: no-name-578086e8-a399-4e47-8e13-a1bc6a3e1c17


TripleIntegratedHierarchicalHybridRecommender: Fusing similarity matrices...
TripleIntegratedHierarchicalHybridRecommender: Similarity fusion complete. Final score combination with gamma=0.11196830894305546.
EvaluatorHoldout: Processed 27064 (100.0%) in 11.92 sec. Users per second: 2271
TripleIntegratedHierarchicalHybridRecommender: Fusing similarity matrices...
TripleIntegratedHierarchicalHybridRecommender: Similarity fusion complete. Final score combination with gamma=0.11196830894305546.
EvaluatorHoldout: Processed 27061 (100.0%) in 11.94 sec. Users per second: 2266
TripleIntegratedHierarchicalHybridRecommender: Fusing similarity matrices...
TripleIntegratedHierarchicalHybridRecommender: Similarity fusion complete. Final score combination with gamma=0.11196830894305546.
EvaluatorHoldout: Processed 27051 (100.0%) in 12.00 sec. Users per second: 2254
TripleIntegratedHierarchicalHybridRecommender: Fusing similarity matrices...
TripleIntegratedHierarchicalHybridRecommender: Similarity f

[I 2025-12-28 11:33:33,037] Trial 0 finished with value: 0.2914214028903219 and parameters: {'alpha': 0.19122291393596239, 'beta': 0.14745930628896467, 'gamma': 0.11196830894305546}. Best is trial 0 with value: 0.2914214028903219.


[0.29253228368772805, 0.2907909140469868, 0.29078671910284765, 0.2911498985635089, 0.29184719905053824]
TripleIntegratedHierarchicalHybridRecommender: Fusing similarity matrices...
TripleIntegratedHierarchicalHybridRecommender: Similarity fusion complete. Final score combination with gamma=0.17545140306924492.
EvaluatorHoldout: Processed 27064 (100.0%) in 12.05 sec. Users per second: 2246
TripleIntegratedHierarchicalHybridRecommender: Fusing similarity matrices...
TripleIntegratedHierarchicalHybridRecommender: Similarity fusion complete. Final score combination with gamma=0.17545140306924492.
EvaluatorHoldout: Processed 27061 (100.0%) in 12.30 sec. Users per second: 2200
TripleIntegratedHierarchicalHybridRecommender: Fusing similarity matrices...
TripleIntegratedHierarchicalHybridRecommender: Similarity fusion complete. Final score combination with gamma=0.17545140306924492.
EvaluatorHoldout: Processed 27051 (100.0%) in 12.09 sec. Users per second: 2237
TripleIntegratedHierarchicalHybr

[I 2025-12-28 11:34:36,564] Trial 1 finished with value: 0.2907401951225731 and parameters: {'alpha': 0.18688871279839747, 'beta': 0.11914820857721269, 'gamma': 0.17545140306924492}. Best is trial 0 with value: 0.2914214028903219.


...

[0.29239570106589063, 0.29076889293401875, 0.29087164232677515, 0.29111640511344994, 0.2917045033598945]
TripleIntegratedHierarchicalHybridRecommender: Fusing similarity matrices...
TripleIntegratedHierarchicalHybridRecommender: Similarity fusion complete. Final score combination with gamma=0.11058529625837912.
EvaluatorHoldout: Processed 27064 (100.0%) in 13.30 sec. Users per second: 2035
TripleIntegratedHierarchicalHybridRecommender: Fusing similarity matrices...
TripleIntegratedHierarchicalHybridRecommender: Similarity fusion complete. Final score combination with gamma=0.11058529625837912.
EvaluatorHoldout: Processed 27061 (100.0%) in 12.84 sec. Users per second: 2108
TripleIntegratedHierarchicalHybridRecommender: Fusing similarity matrices...
TripleIntegratedHierarchicalHybridRecommender: Similarity fusion complete. Final score combination with gamma=0.11058529625837912.
EvaluatorHoldout: Processed 27051 (100.0%) in 13.02 sec. Users per second: 2078
TripleIntegratedHierarchicalHyb

[I 2025-12-28 13:20:05,604] Trial 98 finished with value: 0.2914468486656812 and parameters: {'alpha': 0.1698186181513036, 'beta': 0.08254860183676609, 'gamma': 0.11058529625837912}. Best is trial 22 with value: 0.2915903594832138.


[0.2924972357492318, 0.2907979735718597, 0.29078426698318127, 0.2913511500814595, 0.2918036169426738]
TripleIntegratedHierarchicalHybridRecommender: Fusing similarity matrices...
TripleIntegratedHierarchicalHybridRecommender: Similarity fusion complete. Final score combination with gamma=0.12917983555012938.
EvaluatorHoldout: Processed 27064 (100.0%) in 13.10 sec. Users per second: 2067
TripleIntegratedHierarchicalHybridRecommender: Fusing similarity matrices...
TripleIntegratedHierarchicalHybridRecommender: Similarity fusion complete. Final score combination with gamma=0.12917983555012938.
EvaluatorHoldout: Processed 27061 (100.0%) in 13.01 sec. Users per second: 2080
TripleIntegratedHierarchicalHybridRecommender: Fusing similarity matrices...
TripleIntegratedHierarchicalHybridRecommender: Similarity fusion complete. Final score combination with gamma=0.12917983555012938.
EvaluatorHoldout: Processed 27051 (100.0%) in 13.14 sec. Users per second: 2058
TripleIntegratedHierarchicalHybrid

[I 2025-12-28 13:21:13,291] Trial 99 finished with value: 0.2915124220330789 and parameters: {'alpha': 0.2134597689381168, 'beta': 0.07384029990328188, 'gamma': 0.12917983555012938}. Best is trial 22 with value: 0.2915903594832138.


[0.29251594442885026, 0.2908518849984456, 0.29103047444152236, 0.2914962818055612, 0.29166752449101524]


# Da qui inizia il training su URM_all

In [11]:
return

SyntaxError: 'return' outside function (3438313781.py, line 1)

In [ ]:
best_alpha_test = 0.9874643151475879
best_beta_test = 0.8761805316137236
#Trial 188 finished with value: 0.2909859712486159 and parameters: {'alpha': 0.9874643151475879, 'beta': 0.8761805316137236}.

In [ ]:
recommender_knn_f = ItemKNNCFRecommender(URM_all)
recommender_knn_f.fit(**KNN_params)

In [ ]:
recommender_SLIM_f = SLIMElasticNetRecommender(URM_all)
recommender_SLIM_f.fit(**SLIM_params)

In [ ]:
# Train the final model 
new_similarity = (1 - best_alpha_test) * csr_matrix(recommender_knn_f.W_sparse) + best_alpha_test * recommender_SLIM_f.W_sparse 

hybridrecommender_object_f = ItemKNNCustomSimilarityRecommender(URM_all)
hybridrecommender_object_f.fit(new_similarity)

In [ ]:
als_recommender_f = FeatureCombinedImplicitALSRecommender(URM_all)
als_recommender_f.fit(**IALS_params)

In [ ]:
recommenders = [hybridrecommender_object_f, als_recommender_f]

linear_comb_rec_f = GeneralizedLinearCoupleHybridRecommender(URM_all, recommenders)
linear_comb_rec_f.fit(best_beta_test)

In [ ]:
import time

start_time = time.time()
results = []
for user_id in df_test_user["user_id"].tolist():
        recommendations = linear_comb_rec_f.recommend(user_id, cutoff=20)       
       
        results.append((user_id, ' '.join(map(str, recommendations))))

df_recommendations = pd.DataFrame(results, columns=["user_id", "item_list"])
print(df_recommendations)
df_recommendations.to_csv("recommendations_m2_knn_1.csv", index=False)

end_time = time.time()